In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/260323').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]

In [3]:
# histograms
hists = {}
sources = []
for sgnl, bg in zip(acqs[1::2], acqs[2::2]):
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    sources.append(src)
    print(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    hist_bg = ROOT.TH1F(f"{src} Background", "", BITS12, 0 , BITS12)
    hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    hist_clean = hist_sgnl.Clone(f"{src} Clean")
    hist_clean.Add(hist_bg, -1)
    for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl
    hists[f"{src}_BG"] = hist_bg
    hists[f"{src}_clean"] = hist_clean
    del hist_sgnl, hist_bg
#print(hists)

Cs-137
Na22
Co-60
Am-241


In [4]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)
cv.Divide(2,2)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for idx, (src, lg) in enumerate(zip(sources, lgds)):   
    cv.cd(idx+1)
    
    sgnl = hists[src]
    bg = hists[src+'_BG']
    clean = hists[src+'_clean']
    
    lg.AddEntry(sgnl, "Signal")
    lg.AddEntry(bg, "Background")
    lg.AddEntry(clean, "Subtracted")
    sgnl.SetLineColor(colors[0])
    bg.SetLineColor(colors[1])
    clean.SetLineColor(colors[2])
    sgnl.SetTitle(src)
    sgnl.Draw("hist")
    bg.Draw("sames hist")
    clean.Draw("sames hist")
    ROOT.gPad.Update()
    for i, sp in enumerate([sgnl, bg, clean]):
        st = sp.FindObject("stats")
        # print(type(st))
        st.SetY1NDC(Yinit - i*0.08)
        st.SetY2NDC(0.1)
    lg.Draw()
    cv.cd(idx+1).SetLogy()
    cv.Draw()




In [7]:
#Fitting all the visible peaks. Values around peak input manually

Cs137_fit = ROOT.TF1("Cs137_fit", "gaus", 120,250)

Co60a_fit = ROOT.TF1("Co60a_fit", "gaus", 200,350)
Co60b_fit = ROOT.TF1("Co60b_fit", "gaus", 450,500)

Na22a_fit = ROOT.TF1("Na22a_fit", "gaus", 150,200)
Na22b_fit = ROOT.TF1("Na22b_fit", "gaus", 320,420)
Na22c_fit = ROOT.TF1("Na22c_fit", "gaus", 450,520)

fits = [Cs137_fit, Co60a_fit, Co60b_fit, Na22a_fit, Na22b_fit, Na22c_fit]

#Fit to histogram and find mean value of peak
hists['Cs-137_clean'].Fit(Cs137_fit, "R+S")
hists['Co-60_clean'].Fit(Co60a_fit, "R+S")
hists['Co-60_clean'].Fit(Co60b_fit, "R+S")
hists['Na22_clean'].Fit(Na22a_fit, "R+S")
hists['Na22_clean'].Fit(Na22b_fit, "R+S")
hists['Na22_clean'].Fit(Na22c_fit, "R+S")

Cs137_mean = Cs137_fit.GetParameter('Mean')
Co60a_mean = Co60a_fit.GetParameter('Mean')
Co60b_mean = Co60b_fit.GetParameter('Mean')
Na22a_mean = Na22a_fit.GetParameter('Mean')
Na22b_mean = Na22b_fit.GetParameter('Mean')
Na22c_mean = Na22c_fit.GetParameter('Mean')

channels = [Na22a_mean, Na22b_mean, Na22c_mean]
energies = Na22_MeV
channels = np.array(channels, dtype='float64')
energies = np.array(energies, dtype='float64')

g = ROOT.TGraph(len(channels), channels, energies)

f = ROOT.TF1("calib", "pol1", min(channels), max(channels))
g.Fit(f)

a = f.GetParameter(1)
b = f.GetParameter(0)
print(a)
print(b)








ROOT.gPad.Modified()
ROOT.gPad.Update()

#lg.Draw()
#cv.Draw()

0.004280158214429103
-0.2655932438428593
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      196.518
NDf                       =          127
Edm                       =  2.29967e-07
NCalls                    =           65
Constant                  =      2459.66   +/-   7.74946     
Mean                      =      183.993   +/-   0.078652    
Sigma                     =      29.2438   +/-   0.0713573    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      356.977
NDf                       =          147
Edm                       =  1.20533e-06
NCalls                    =           64
Constant                  =      1361.97   +/-   6.38965     
Mean                      =       275.54   +/-   0.144272    
Sigma                     =      32.7382   +/-   0.141925     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                  

In [14]:
if ROOT.gROOT.FindObject('cv_cal'):
    ROOT.gROOT.FindObject('cv_cal').Close()

    
hist = hists['Na22_clean']

emin = a * hist.GetXaxis().GetXmin() + b
emax = a * hist.GetXaxis().GetXmax() + b

h_cal = ROOT.TH1F("h_cal", "Calibrated Spectrum",
             hist.GetNbinsX(), emin, emax)

for i in range(1, hist.GetNbinsX() + 1):
    counts = hist.GetBinContent(i)
    h_cal.SetBinContent(i, counts)

h_cal.GetXaxis().SetTitle("Energy (MeV)")


Yinit = 0.82 # For stat boxes

cv_cal = ROOT.TCanvas('cv_cal', 'cv_cal', 1600, 1200)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]



Na22a_fit_cal = ROOT.TF1("Na22a_fit_cal", "gaus", 0.3, 0.6)
Na22b_fit_cal = ROOT.TF1("Na22b_fit_cal", "gaus", 1.1 , 1.5)
Na22c_fit_cal = ROOT.TF1("Na22c_fit_cal", "gaus", 1.5, 1.9)
h_cal.Fit(Na22a_fit_cal, "R+S")
h_cal.Fit(Na22b_fit_cal, "R+S")
h_cal.Fit(Na22c_fit_cal, "R+S")

h_cal.Draw("hist")
cv_cal.cd(idx+1).SetLogy()

cv_cal.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      137.647
NDf                       =           67
Edm                       =  2.07898e-06
NCalls                    =           84
Constant                  =      189.854   +/-   2.40817     
Mean                      =     0.492681   +/-   0.00341376  
Sigma                     =     0.142338   +/-   0.00437932   	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      155.228
NDf                       =           91
Edm                       =   3.6702e-08
NCalls                    =           96
Constant                  =      613.942   +/-   3.98488     
Mean                      =      1.32007   +/-   0.00148807  
Sigma                     =     0.180551   +/-   0.00242551   	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      372.407
NDf                   

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
